# FLUX outpainting — E2E validation (all-sides extend)

Training-free outpainting: paste the source on a larger canvas and inpaint only the new margins with
FLUX.1-Fill. Steps: **raw-path spike** (proves the canvas+mask mechanism against `FluxFillPipeline`
directly) → publish PRIVATE → **modular load** via `trust_remote_code` → validate the three claims:
**(a)** the interior is pixel-equal to the source, **(b)** the new margins are coherent and blend at the
seam, **(c)** a no-op control (`mask_feather=0` + zero margins is rejected rather than returning garbage).

Runtime: A100 · `HUGGINGFACE_TOKEN` · accept **FLUX.1-dev** AND **FLUX.1-Fill-dev** licenses.


## 1 · Install + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf


In [ ]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
import torch
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))


## 2 · Source photo

In [ ]:
import requests, numpy as np
from PIL import Image
from io import BytesIO
from IPython.display import display
from torchvision import transforms

url = "https://raw.githubusercontent.com/nftblackmagic/catvton-flux/main/example/person/1.jpg"
src = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB").resize((512, 768))
src.save("e2e_src.png")
print("source:", src.size); display(src.resize((256, 384)))


## 3 · Milestone A — raw-path spike (FLUX.1-Fill, no modular wrapper)

De-risk the mechanism first (CONVENTIONS: spike before wrapping): build the padded canvas and margin mask
by hand and call `FluxFillPipeline` directly. Note the **[0,1]** image range — current `FluxFillPipeline`
normalizes internally; passing [-1,1] corrupts the conditioning.

In [ ]:
from diffusers import FluxTransformer2DModel, FluxFillPipeline
FILL, BASE = "black-forest-labs/FLUX.1-Fill-dev", "black-forest-labs/FLUX.1-dev"
tr = FluxTransformer2DModel.from_pretrained(FILL, subfolder="transformer", torch_dtype=DT)
raw_pipe = FluxFillPipeline.from_pretrained(BASE, transformer=tr, torch_dtype=DT).to(DEV)

# the block's geometry, reproduced by hand for the spike
M, F = 128, 8                       # margin per side, feather
W, H = src.size
CW, CH = W + 2 * M, H + 2 * M       # 768x1024, already multiples of 8
to01 = transforms.ToTensor()
canvas = torch.full((3, CH, CW), 0.5)
canvas[:, M:M + H, M:M + W] = to01(src)
mask = torch.ones(1, CH, CW)
mask[:, M + F:M + H - F, M + F:M + W - F] = 0   # keep the eroded interior, repaint margins + feather band

g = torch.Generator(DEV).manual_seed(0)
raw = raw_pipe(height=CH, width=CW, image=canvas, mask_image=mask,
               num_inference_steps=30, guidance_scale=30, generator=g, max_sequence_length=512,
               prompt="the same scene continuing beyond the frame, consistent lighting").images[0]
raw_out = np.asarray(raw).copy()
raw_out[M + F:M + H - F, M + F:M + W - F] = np.asarray(src)[F:H - F, F:W - F]
print("[MILESTONE A] raw outpaint OK", raw_out.shape)
display(Image.fromarray(raw_out).resize((384, 512)))
del raw_pipe, tr; import gc; gc.collect(); torch.cuda.empty_cache()


## 4 · Milestone B — publish PRIVATE + load modular (`trust_remote_code`)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(); REPO = "remyxai/outpaint-flux-modular"
api.create_repo(REPO, private=True, repo_type="model", exist_ok=True)
for f in ["block.py", "modular_config.json", "modular_model_index.json"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO)
print("published:", api.list_repo_files(REPO))

from diffusers import ModularPipeline
pipe = ModularPipeline.from_pretrained(REPO, trust_remote_code=True)
print("loaded block:", type(pipe.blocks).__name__)
assert type(pipe.blocks).__name__ == "OutpaintBlock"
pipe.load_components(dtype=DT); pipe.to(DEV)


## 5 · Milestone C — all-sides extend + the three claims

| claim | check |
|---|---|
| **a. interior untouched** | the eroded interior is **pixel-equal** to the source (`np.array_equal`) |
| **b. margins coherent / seamless** | every new pixel differs from the neutral 0.5 fill (it was repainted, not left blank), and the gradient discontinuity across the old border is no worse than a natural image edge |
| **c. no-op control** | zero margins raises instead of returning the input |


In [ ]:
g = torch.Generator(DEV).manual_seed(0)
out_img = pipe(image=src, prompt="the same scene continuing beyond the frame, consistent lighting",
               left=128, right=128, top=128, bottom=128, mask_feather=8,
               num_inference_steps=30, guidance_scale=30, generator=g).images[0]
out = np.asarray(out_img)
S, M, F = np.asarray(src), 128, 8
interior = out[M + F:M + H - F, M + F:M + W - F]
orig_interior = S[F:H - F, F:W - F]

# (a) interior bit-exact
assert interior.shape == orig_interior.shape
assert np.array_equal(interior, orig_interior), "interior must be pixel-equal to the source"
print(f"[a] interior pixel-equal: {interior.shape[1]}x{interior.shape[0]} px, max abs diff = "
      f"{int(np.abs(interior.astype(int) - orig_interior.astype(int)).max())}")

# (b) the margins were actually repainted (not left as neutral fill) and the seam is not a hard edge
margins = np.concatenate([out[:M].reshape(-1, 3), out[-M:].reshape(-1, 3),
                          out[M:-M, :M].reshape(-1, 3), out[M:-M, -M:].reshape(-1, 3)])
assert np.abs(margins - 127.5).max() > 1.0, "margins were never repainted (still the neutral fill)"
print(f"[b] margins repainted: mean |dev from fill| = {np.abs(margins - 127.5).mean():.1f}")

# seam metric: gradient magnitude just inside vs just across the old border. A visible seam would
# show a spike at the border row/col relative to the image's own natural edge strength.
def grad_cols(a): return np.abs(np.diff(a.astype(float), axis=1)).mean()
border = grad_cols(out[M + 2:H + M - 2, M + W - 3:M + W + 3])       # across the old right border
inside = grad_cols(out[M + 2:H + M - 2, M + W // 2 - 3:M + W // 2 + 3])   # mid-image, natural edges
print(f"[b] seam check: border-adjacent gradient {border:.2f} vs mid-image {inside:.2f} "
      f"(ratio {border / inside:.2f})")
assert border / inside < 3.0, "hard seam at the original/new boundary"

# (c) no-op control: zero margins must raise, not silently return the input
try:
    pipe(image=src, prompt="x", left=0, right=0, top=0, bottom=0)
    raise AssertionError("zero margins were accepted")
except ValueError as e:
    print(f"[c] no-op control raised as expected: {e}")

out_img.save("e2e_out.png")
side = Image.new("RGB", (W + CW // 4, H), "white")
side.paste(src, (0, 0)); side.paste(out_img.resize((CW // 4, H)), (W, 0))
side.save("outpaint_demo.png")
print("source | outpainted"); display(side.resize((512, 384)))


## Verdict

`loaded block: OutpaintBlock` + **(a)** pixel-equal interior + **(b)** repainted, non-seamed margins +
**(c)** the no-op control raising = the modular outpainting pipeline works end-to-end and matches the
raw-path spike. Then: publish public, link the Colab, add to the collection.